# 01. EDA: как SaaS-метрики называют в реальных проспектах SEC

**Зачем.** Система извлекает утверждения из текстов компаний. Синтетические тексты пишет наш генератор,
поэтому нужно реальное сравнение: какими словами компании называют ARR, NRR, churn и концентрацию,
насколько эти слова специфичны для софтверных компаний и из каких проспектов будет собран корпус предложений.

**Что доступно без контакта в User-Agent.** Полнотекстовый поиск EDGAR (`efts.sec.gov`) отвечает на запросы
с общим User-Agent `dd-verifier-student-project research`. Он возвращает число документов с точной фразой и
агрегаты по отраслевым кодам SIC, но не текст. Архив документов `www.sec.gov` без контакта в User-Agent отвечает
403, поэтому корпус предложений `data/reference/sec_kpi_claims.csv` пока не собран
(`scripts/fetch_sec_claims.py`, контакт задаётся только через переменную окружения `SEC_USER_AGENT`).

**Что здесь есть.**
1. Частоты 21 термина в S-1 и 424B4 за 2019–2025: все отрасли и только SIC 737x (программное обеспечение
   и компьютерные услуги).
2. 50 проспектов SaaS, отобранных для будущего корпуса: формы, годы, коды SIC.

Агрегаты сохраняются в `data/reference/`, поэтому ноутбук воспроизводится офлайн. Выводы — в
`reports/eda/01_sec_claims.md`.

In [ ]:
import hashlib
import importlib.util
import json
import os
import sys
from pathlib import Path
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

spec = importlib.util.spec_from_file_location(
    "fetch_sec_claims", Path("scripts/fetch_sec_claims.py")
)
fsc = importlib.util.module_from_spec(spec)
sys.modules["fetch_sec_claims"] = fsc
spec.loader.exec_module(fsc)

REF = Path("data/reference")
FIG = Path("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)
TERMS_CSV = REF / "sec_efts_term_counts.csv"
FILINGS_CSV = REF / "sec_filings_selected.csv"
CACHE = Path("data/external/sec/terms_sic")
SOFTWARE_SIC = "737"  # 7370–7379: программное обеспечение и компьютерные услуги

C = ["#2a78d6", "#eb6834", "#6250d6", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#e34948"]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK2,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "font.size": 9,
        "legend.frameon": False,
    }
)


def save(fig, name: str) -> None:
    path = FIG / f"01_{name}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    print(f"{path}: {path.stat().st_size / 1024:.0f} KB")

## 1. Частоты терминов: все отрасли и SIC 737x

Для каждого термина один запрос к `efts.sec.gov` (точная фраза, формы S-1 и 424B4, подача 2019-01-01…2025-12-31).
Из ответа берём `hits.total` (все отрасли) и агрегат `sic_filter`: сумма корзин с кодом 737x. Агрегат отдаёт
30 самых частых кодов SIC, остальное — в `sum_other_doc_count`, поэтому число для 737x — нижняя граница.
Единица счёта — документ (файл заявки), а не компания: одна компания подаёт S-1, несколько S-1/A и 424B4.

In [ ]:
TERMS = [
    # (термин, группа, метрика онтологии)
    ("annual recurring revenue", "ARR / MRR", "arr"),
    ("monthly recurring revenue", "ARR / MRR", "mrr"),
    ("annualized run-rate", "ARR / MRR", "arr (run-rate)"),
    ("net revenue retention", "NRR", "nrr"),
    ("net dollar retention", "NRR", "nrr"),
    ("dollar-based net retention rate", "NRR", "nrr"),
    ("dollar-based net expansion rate", "NRR", "nrr"),
    ("net retention rate", "NRR", "nrr"),
    ("gross revenue retention", "GRR", "grr"),
    ("gross dollar retention", "GRR", "grr"),
    ("gross retention rate", "GRR", "grr"),
    ("logo churn", "Churn", "logo_churn"),
    ("customer churn", "Churn", "logo_churn"),
    ("logo retention", "Churn", "logo_churn"),
    ("average contract value", "ACV", "acv"),
    ("annual contract value", "ACV", "acv"),
    ("ten largest customers", "Концентрация", "top10_concentration"),
    ("CAC payback", "вне онтологии", "cac_payback"),
    ("customer acquisition cost", "вне онтологии", "cac_payback"),
    ("lifetime value", "вне онтологии", "ltv_to_cac"),
    ("burn multiple", "вне онтологии", "burn_multiple"),
]


def efts_query(client, term: str) -> dict:
    CACHE.mkdir(parents=True, exist_ok=True)
    cache = CACHE / f"{hashlib.sha1(term.encode()).hexdigest()[:16]}.json"
    if cache.exists():
        return json.loads(cache.read_text())
    url = (
        f"{fsc.SEARCH_URL}?q={quote(chr(34) + term + chr(34))}&forms={fsc.FORMS}"
        f"&dateRange=custom&startdt={fsc.START_DATE}&enddt={fsc.END_DATE}"
    )
    data = client.get(url).json()
    keep = {"total": data["hits"]["total"], "sic_filter": data["aggregations"]["sic_filter"]}
    cache.write_text(json.dumps(keep))
    return keep


def term_row(term: str, group: str, metric: str, data: dict) -> dict:
    buckets = data["sic_filter"]["buckets"]
    soft = sum(b["doc_count"] for b in buckets if b["key"].startswith(SOFTWARE_SIC))
    top = max(buckets, key=lambda b: b["doc_count"]) if buckets else {"key": "", "doc_count": 0}
    return {
        "term": term,
        "group": group,
        "ontology_metric": metric,
        "n_documents": int(data["total"]["value"]),
        "n_documents_sic737x": int(soft),
        "top_sic": top["key"],
        "top_sic_documents": int(top["doc_count"]),
        "sic_other_documents": int(data["sic_filter"].get("sum_other_doc_count", 0)),
    }


have = pd.read_csv(TERMS_CSV) if TERMS_CSV.exists() else None
if have is not None and "n_documents_sic737x" in have.columns:
    TERM_COUNTS = have  # офлайн: агрегаты уже в репозитории
else:
    client = fsc.SecClient(os.environ.get("SEC_USER_AGENT", fsc.DEFAULT_UA))
    TERM_COUNTS = pd.DataFrame([term_row(t, g, m, efts_query(client, t)) for t, g, m in TERMS])
    TERM_COUNTS["forms"] = "S-1,424B4"
    TERM_COUNTS["filed"] = f"{fsc.START_DATE}..{fsc.END_DATE}"
    TERM_COUNTS["queried_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
    REF.mkdir(parents=True, exist_ok=True)
    TERM_COUNTS.to_csv(TERMS_CSV, index=False)
    print("запросов к EFTS:", client.n_requests)

TC = TERM_COUNTS.copy()
TC["share_sic737x"] = TC["n_documents_sic737x"] / TC["n_documents"].where(TC["n_documents"] > 0)
print(
    TC[["group", "term", "n_documents", "n_documents_sic737x", "share_sic737x", "top_sic"]]
    .round(2)
    .to_string(index=False)
)
by_group = TC.groupby("group")[["n_documents", "n_documents_sic737x"]].sum()
by_group["share_sic737x"] = by_group["n_documents_sic737x"] / by_group["n_documents"]
print()
print(by_group.sort_values("n_documents_sic737x", ascending=False).round(2).to_string())

In [ ]:
# Синонимы внутри группы: доля каждого названия среди документов SIC 737x (сумма по названиям группы)
syn = TC[TC["group"].isin(["NRR", "GRR", "Churn", "ARR / MRR", "ACV"])].copy()
syn["share_in_group"] = syn["n_documents_sic737x"] / syn.groupby("group")[
    "n_documents_sic737x"
].transform("sum")
print(
    syn.sort_values(["group", "share_in_group"], ascending=[True, False])[
        ["group", "term", "n_documents_sic737x", "share_in_group"]
    ]
    .round(2)
    .to_string(index=False)
)
nrr = syn[syn["group"] == "NRR"]
print(
    f"\nNRR: {len(nrr)} названий; самое частое покрывает "
    f"{nrr['share_in_group'].max():.0%} упоминаний в SIC 737x"
)

In [ ]:
tc = TC.sort_values(["group", "n_documents_sic737x"], ascending=[True, True]).reset_index(drop=True)
groups = list(dict.fromkeys(tc["group"]))
gcolor = {g: C[i % len(C)] for i, g in enumerate(groups)}
fig, axes = plt.subplots(
    1, 2, figsize=(12.5, 5.6), sharey=True, gridspec_kw={"width_ratios": [1.6, 1]}
)
ax = axes[0]
y = np.arange(len(tc))
ax.barh(y, tc["n_documents"], color="#dcdad3", height=0.75, label="все отрасли")
ax.barh(
    y,
    tc["n_documents_sic737x"],
    color=[gcolor[g] for g in tc["group"]],
    height=0.45,
    label="SIC 737x (ПО)",
)
for i, r in tc.iterrows():
    ax.text(
        r["n_documents"],
        i,
        f" {r['n_documents_sic737x']} / {r['n_documents']}",
        va="center",
        fontsize=7,
        color=INK2,
    )
ax.set_yticks(y, tc["term"])
ax.set_xlim(0, tc["n_documents"].max() * 1.22)
ax.set_xlabel("документов S-1 / 424B4 за 2019–2025 с точной фразой")
ax.set_title("Сколько проспектов используют термин: ПО против всех отраслей", loc="left")
ax.grid(axis="y", visible=False)
handles = [plt.Rectangle((0, 0), 1, 1, color="#dcdad3")] + [
    plt.Rectangle((0, 0), 1, 1, color=gcolor[g]) for g in groups
]
ax.legend(
    handles, ["все отрасли"] + [f"SIC 737x: {g}" for g in groups], fontsize=7, loc="lower right"
)
ax = axes[1]
share = (tc["share_sic737x"] * 100).fillna(0)
ax.barh(y, share, color=[gcolor[g] for g in tc["group"]], height=0.6)
for i, v in enumerate(share):
    ax.text(v, i, f" {v:.0f}%", va="center", fontsize=7, color=INK2)
ax.axvline(50, color=MUTED, lw=0.8, ls=":")
ax.set_xlim(0, 110)
ax.set_xlabel("доля документов с SIC 737x, % (нижняя граница)")
ax.set_title("Насколько термин «софтверный»", loc="left")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save(fig, "terms_by_industry")

## 2. Отобранные проспекты для корпуса предложений

`scripts/fetch_sec_claims.py` ищет 6 фраз (ARR и пять названий NRR/GRR), оставляет основные документы заявки
(S-1, S-1/A, 424B4) компаний с первым кодом SIC 737x и берёт по одной заявке на компанию: 424B4 предпочтительнее
S-1/A и S-1. Компании ранжируются по числу разных запросов, на которые они нашлись. Метаданные 50 отобранных
заявок копируем в `data/reference/sec_filings_selected.csv` (без текста документов).

In [ ]:
EXT = Path("data/external/sec/filings_selected.csv")
COLS = ["cik", "company", "sic", "form", "filing_date", "n_queries", "url"]
if EXT.exists():
    FIL = pd.read_csv(EXT, dtype={"sic": str})[COLS]
    FIL.to_csv(FILINGS_CSV, index=False)
else:
    FIL = pd.read_csv(FILINGS_CSV, dtype={"sic": str})
FIL["year"] = FIL["filing_date"].str[:4].astype(int)
print("заявок:", len(FIL), "| компаний:", FIL["cik"].nunique())
print("формы:", FIL["form"].value_counts().to_dict())
print("SIC:", FIL["sic"].value_counts().to_dict())
print("годы:", FIL["year"].value_counts().sort_index().to_dict())
print("на сколько разных запросов нашлась компания:", FIL["n_queries"].value_counts().to_dict())

In [ ]:
years = list(range(2019, 2026))
forms = ["424B4", "S-1/A", "S-1"]
tab = pd.crosstab(FIL["year"], FIL["form"]).reindex(index=years, columns=forms, fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
bottom = np.zeros(len(years))
for k, f in enumerate(forms):
    ax.bar(years, tab[f], bottom=bottom, color=C[k], width=0.65, label=f)
    bottom += tab[f].to_numpy()
for x, v in zip(years, bottom, strict=True):
    ax.text(x, v, f"{int(v)}", ha="center", va="bottom", fontsize=8, color=INK2)
ax.set_ylabel("заявок")
ax.set_title("50 отобранных проспектов SaaS по году подачи", loc="left")
ax.legend(fontsize=8)
ax.grid(axis="x", visible=False)
ax = axes[1]
sic = FIL["sic"].value_counts().sort_values()
ax.barh(sic.index, sic.values, color=C[0], height=0.6)
for i, v in enumerate(sic.values):
    ax.text(v, i, f" {v}", va="center", fontsize=8, color=INK2)
ax.set_xlabel("заявок")
ax.set_title("Код SIC", loc="left")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save(fig, "filings_selected")

## 3. Что остаётся сделать с корпусом предложений

Когда автор запустит `SEC_USER_AGENT="<имя> <e-mail>" uv run python scripts/fetch_sec_claims.py` (контакт только
в окружении), появится `data/reference/sec_kpi_claims.csv`. Тогда этот ноутбук дополняется разделом по предложениям:
доли утверждений по метрикам, квалификаторы («more than», «approximately»), типы периодов, числовые форматы,
доля предложений-определений и точность экстрактора на ручной выборке. Раздел 8 EDA 03 уже умеет сравнивать
синтетику с этим файлом, если он есть.